# 150M-param GPT: pretraining + base-model evaluation (Local / VS Code / RTX 5050 8GB)

Reference architecture: Andrej Karpathy's nanoGPT (GPT-2 style decoder-only transformer).

**This version runs locally in VS Code** (not Google Colab), tuned for:
- CPU: AMD Ryzen 7
- RAM: 24 GB DDR5
- GPU: NVIDIA GeForce RTX 5050 (8 GB VRAM, Blackwell architecture)
- Storage: 1 TB SSD
- OS: Windows 11

**Pipeline:**
1. Model definition from scratch, including RMSNorm, GQA, RoPE, QK-Norm, and a SwiGLU MoE
2. Pretraining on a 70/30 mixture of FineWeb-Edu and FineWeb
3. Validation on the WikiText-103 validation split
4. Base-model text-continuation sanity checks

**nanoGPT-style features included:**
- Flash attention through `scaled_dot_product_attention`, weight tying, and scaled residual initialization
- BF16 on supported GPUs, with FP16 plus GradScaler fallback on older CUDA GPUs
- TF32 matmuls when supported
- Cosine learning-rate schedule with warmup, AdamW, fused AdamW when available
- Gradient accumulation, gradient clipping, and gradient checkpointing
- Token-throughput, MFU, gradient, and peak-VRAM diagnostics
- `latest.pt` and `best.pt` pretraining checkpoints with automatic resume
- Base-model generation with top-k and nucleus (top-p) sampling

**Checkpointing:** the pretraining loop saves checkpoints to the local project folder every 1,000 optimizer steps and resumes from `latest.pt` when available.

**Expectations:** this is a from-scratch 150M-scale model, not GPT-4. It is intended for learning and experimentation. A base model produces text continuations; it is not instruction-tuned to answer questions.

**A note on time:** the configured run processes about 5 billion tokens over 152,588 optimizer steps. On an 8 GB GPU, this may take multiple days. Lower `PRETRAIN_MAX_STEPS` for a shorter experiment; checkpointing lets you resume later.

## 1. Setup: local project folder, dependencies, GPU check


In [ ]:
import os
import sys
from pathlib import Path
import torch

# -----------------------------------------------------------------------------
# 1. Directory Configuration
# -----------------------------------------------------------------------------
# Store checkpoints and datasets locally in user's home directory
PROJECT_DIR = Path.home() / 'gpt150m_project'
DATA_DIR = PROJECT_DIR / 'data'
PRETRAIN_CKPT_DIR = PROJECT_DIR / 'checkpoints_pretrain_v4'

# Create all necessary directories safely
for directory in [DATA_DIR, PRETRAIN_CKPT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")

# -----------------------------------------------------------------------------
# 2. PyTorch & GPU Initialization Checks
# -----------------------------------------------------------------------------
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    # Binary conversion (1024^3) yields accurate GiB reported by system drivers
    total_vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {total_vram_gib:.2f} GiB")
    
    # Check compute capability (sm_120 support warning for Blackwell GPUs)
    capability = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {capability[0]}.{capability[1]}")
    
    if total_vram_gib < 7.5:
        print(
            "WARNING: Less than 7.5 GiB VRAM detected. "
            "Consider lowering your batch size or context length to prevent Out-Of-Memory (OOM) errors."
        )
else:
    print(
        "WARNING: No CUDA GPU detected. Training will fall back to CPU and execute very slowly.\n"
        "Ensure PyTorch is installed with CUDA support:\n"
        "  pip install torch --index-url https://download.pytorch.org/whl/cu128"
    )

Project dir: C:\Users\sandyarjun\gpt150m_project
Python version: 3.12.10
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
VRAM: 7.96 GiB
Compute Capability: 12.0


## 2. Model definition (GPT-2 architecture, Karpathy nanoGPT style)

In [2]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.utils.checkpoint
from torch.nn import functional as F

@dataclass
class GPTConfig:
    block_size: int = 2048
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = False
    rope_theta: float = 10000.0
    rope_scaling_factor: float = 1.0
    qk_norm: bool = True
    moe_num_experts: int = 4
    moe_top_k: int = 2
    moe_hidden_dim: int | None = None
    moe_aux_loss_coeff: float = 0.01
    gradient_checkpointing: bool = False

    def __post_init__(self):
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.n_head % self.n_kv_head != 0:
            raise ValueError("n_head must be divisible by n_kv_head")
        if self.moe_num_experts < 1:
            raise ValueError("moe_num_experts must be >= 1")
        if self.moe_top_k < 1 or self.moe_top_k > self.moe_num_experts:
            raise ValueError(f"moe_top_k must be between 1 and {self.moe_num_experts}")
        if self.moe_hidden_dim is not None and self.moe_hidden_dim <= 0:
            raise ValueError("moe_hidden_dim must be > 0")
        if self.rope_scaling_factor <= 0:
            raise ValueError("rope_scaling_factor must be > 0")

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        variance = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, theta=10000.0, scaling_factor=1.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float() / scaling_factor
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
    def forward(self, position_ids):
        return self.cos_cached[position_ids], self.sin_cached[position_ids]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    cos, sin = cos[None, None, :, :], sin[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class KVCache:
    def __init__(self, batch_size, n_kv_head, max_seq_len, head_dim, device, dtype):
        self.k = torch.zeros((batch_size, n_kv_head, max_seq_len, head_dim), device=device, dtype=dtype)
        self.v = torch.zeros((batch_size, n_kv_head, max_seq_len, head_dim), device=device, dtype=dtype)
        self.length = 0

    def update(self, k, v):
        if k.size(0) != self.k.size(0):
            raise ValueError(f"KV cache batch size mismatch: cache={self.k.size(0)}, input={k.size(0)}")
        if k.size(1) != self.k.size(1):
            raise ValueError(f"KV cache head mismatch: cache={self.k.size(1)}, input={k.size(1)}")
        if k.size(3) != self.k.size(3):
            raise ValueError(f"KV cache head_dim mismatch: cache={self.k.size(3)}, input={k.size(3)}")
        if v.shape != k.shape:
            raise ValueError(f"K/V shape mismatch: K={tuple(k.shape)}, V={tuple(v.shape)}")

        seqlen = k.size(2)
        if self.length + seqlen > self.k.size(2):
            raise ValueError(f"KV Cache overflow: {self.length + seqlen} > {self.k.size(2)}")

        self.k[:, :, self.length:self.length+seqlen].copy_(k)
        self.v[:, :, self.length:self.length+seqlen].copy_(v)
        self.length += seqlen
        return self.k[:, :, :self.length], self.v[:, :, :self.length]

class SwiGLUExpert(nn.Module):
    def __init__(self, config, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.w3 = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class SparseMoE(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_experts = config.moe_num_experts
        hidden_dim = config.moe_hidden_dim or int(8 * config.n_embd / 3)
        self.top_k = config.moe_top_k
        self.router = nn.Linear(config.n_embd, self.n_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLUExpert(config, hidden_dim) for _ in range(self.n_experts)])
    def forward(self, x, return_aux_loss=False):
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        logits = self.router(x_flat)
        gate_probs = F.softmax(logits, dim=-1)
        weights, indices = torch.topk(gate_probs, self.top_k, dim=-1)
        weights = weights / weights.sum(dim=-1, keepdim=True)

        aux_loss = None
        if return_aux_loss:
            importance = gate_probs.mean(dim=0)
            routing_mask = F.one_hot(indices, num_classes=self.n_experts).float()
            load = routing_mask.sum(dim=(0, 1)) / (x_flat.size(0) * self.top_k)
            aux_loss = self.n_experts * torch.sum(importance * load)

        out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (indices == i).any(dim=-1)
            if mask.any():
                expert_out = expert(x_flat[mask])
                w = (indices[mask] == i).float() * weights[mask]
                out[mask] += expert_out * w.sum(dim=-1, keepdim=True)
        return out.view(B, T, C), aux_loss

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head, self.n_kv_head = config.n_head, config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.kv_repeat = self.n_head // self.n_kv_head
        self.qkv_proj = nn.Linear(config.n_embd, (config.n_head + 2*config.n_kv_head) * self.head_dim, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.rope = RotaryEmbedding(self.head_dim, config.block_size, config.rope_theta, config.rope_scaling_factor)
        self.q_norm = RMSNorm(self.head_dim) if config.qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.head_dim) if config.qk_norm else nn.Identity()
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, C = x.shape
        if use_cache and past_kv is not None and T > 1:
            raise ValueError("Cached decoding currently supports T=1 only.")

        q, k, v = self.qkv_proj(x).split([self.n_head*self.head_dim, self.n_kv_head*self.head_dim, self.n_kv_head*self.head_dim], dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        q, k = self.q_norm(q), self.k_norm(k)
        pos_ids = torch.arange(T, device=x.device) + (past_kv.length if past_kv else 0)
        cos, sin = self.rope(pos_ids)
        q, k = apply_rope(q, k, cos, sin)

        if use_cache:
            if past_kv is None: past_kv = KVCache(B, self.n_kv_head, self.config.block_size, self.head_dim, x.device, x.dtype)
            k, v = past_kv.update(k, v)

        k = k.repeat_interleave(self.kv_repeat, dim=1)
        v = v.repeat_interleave(self.kv_repeat, dim=1)

        is_causal = not (use_cache and past_kv is not None and T == 1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, dropout_p=self.attn_dropout.p if self.training else 0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), past_kv

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = SparseMoE(config)
    def forward(self, x, past_kv=None, use_cache=False, return_aux_loss=False):
        attn_out, present = self.attn(self.ln_1(x), past_kv, use_cache)
        x = x + attn_out
        mlp_out, aux_loss = self.mlp(self.ln_2(x), return_aux_loss=return_aux_loss)
        x = x + mlp_out
        return x, present, aux_loss

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = RMSNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight') or pn.endswith('w3.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding: n_params -= self.transformer.wte.weight.numel()
        return n_params

    def forward(self, idx, targets=None, past_kvs=None, use_cache=False):
        B, T = idx.shape
        if past_kvs is not None:
            if len(past_kvs) != self.config.n_layer:
                raise ValueError(f"Expected {self.config.n_layer} KV caches, got {len(past_kvs)}.")
            curr_pos = past_kvs[0].length
            if any(kv.length != curr_pos for kv in past_kvs):
                raise ValueError("All KV caches must have the same length.")
        else:
            curr_pos = 0

        if curr_pos + T > self.config.block_size:
             raise ValueError(f"Total length {curr_pos + T} exceeds block size {self.config.block_size}")
        x = self.transformer.drop(self.transformer.wte(idx))
        new_kvs = []
        total_aux_loss = 0

        for i, block in enumerate(self.transformer.h):
            pk = past_kvs[i] if past_kvs else None
            ret_aux = (targets is not None)

            if self.config.gradient_checkpointing and self.training and not use_cache:
                def cp_forward(hidden_states, block=block, ret_aux=ret_aux):
                    res_x, _, res_aux = block(hidden_states, None, False, ret_aux)
                    return res_x, res_aux
                x, block_aux = torch.utils.checkpoint.checkpoint(cp_forward, x, use_reentrant=False)
                if block_aux is not None: total_aux_loss += block_aux
                nk = None
            else:
                x, nk, aux_loss = block(x, pk, use_cache, return_aux_loss=ret_aux)
                if aux_loss is not None: total_aux_loss += aux_loss

            if use_cache: new_kvs.append(nk)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            lm_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = lm_loss + self.config.moe_aux_loss_coeff * total_aux_loss
        return (logits, loss, new_kvs, total_aux_loss) if use_cache else (logits, loss, total_aux_loss)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        past_kvs = None
        for _ in range(max_new_tokens):
            if idx.size(1) >= self.config.block_size:
                break
            idx_input = idx if past_kvs is None else idx[:, -1:]
            logits, _, past_kvs, _ = self(idx_input, past_kvs=past_kvs, use_cache=True)
            logits = logits[:, -1, :]

            if temperature <= 0 or not math.isfinite(temperature):
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None and top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                if top_p is not None and 0.0 < top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    sorted_probs = F.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = False

                    sorted_logits[sorted_indices_to_remove] = -float('Inf')
                    logits = torch.full_like(logits, -float('Inf'))
                    logits.scatter_(1, sorted_indices, sorted_logits)

                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 3. Model config -- 151.5M params (tuned for 8GB VRAM)


In [3]:
# Version 4 config: RMSNorm + GQA + QK-Norm + RoPE + FlashAttention +
# preallocated KV cache + SwiGLU MoE. The expert width preserves the original
# approximately 150M-parameter / 8GB-VRAM training target.
cfg = GPTConfig(
    block_size=1024,
    vocab_size=50257,
    n_layer=12,
    n_head=8,
    n_kv_head=4,          # GQA: two query heads share each KV head
    n_embd=864,
    dropout=0.0,          # modern pretraining default
    bias=False,
    qk_norm=True,
    moe_num_experts=4,
    moe_top_k=2,
    moe_hidden_dim=768,   # practical MoE width for this 8GB target
    gradient_checkpointing=True,  # ON by default for 8GB VRAM targets
)

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# TF32 matmuls: free speedup on Ampere+/Blackwell
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- Precision Selection ---
if device == 'cuda' and torch.cuda.is_bf16_supported():
    ptdtype = torch.bfloat16
    use_grad_scaler = False
else:
    ptdtype = torch.float16 if device == 'cuda' else torch.float32
    use_grad_scaler = (device == 'cuda')

gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'device: {device} ({gpu_name})')
print(f'training dtype: {ptdtype} | GradScaler enabled: {use_grad_scaler}')

# MFU Estimate reference for RTX 5050 (estimate: 26 TFLOPS)
GPU_PEAK_FLOPS = {
    'RTX 5050': 26e12,
    'T4': 65e12,
    'A100': 312e12,
}
PEAK_FLOPS = 1e12
if device == 'cuda':
    PEAK_FLOPS = 30e12
    for key, val in GPU_PEAK_FLOPS.items():
        if key in gpu_name:
            PEAK_FLOPS = val
            break

_tmp_model = GPT(cfg)
params = _tmp_model.get_num_params()
print(f'Model parameters (non-embedding): {params / 1e6:.2f}M')
del _tmp_model

device: cuda (NVIDIA GeForce RTX 5050 Laptop GPU)
training dtype: torch.bfloat16 | GradScaler enabled: False
Model parameters (non-embedding): 122.49M


## 4. Pretraining data

Streams FineWeb-Edu `sample-10BT` and FineWeb `sample-10BT`, mixing them at 70/30 and writing GPT-2 BPE tokens to shuffled `uint16` training shards. WikiText-103 validation text is written separately to `pretrain_val.bin`. The target is 5 billion training tokens and 10 million validation tokens. Data is streamed and written directly to disk, so the full corpus is not accumulated in RAM.

In [4]:
# ============================================================
# V4 PRETRAINING DATA
# FineWeb-Edu 70% + FineWeb 30%
# GPT-2 BPE tokenizer
# True streaming -> uint16 binary shards
#
# Training:
#   FineWeb-Edu sample-10BT : 70%
#   FineWeb sample-10BT     : 30%
#
# Validation:
#   WikiText-103 validation split
#
# This version does NOT accumulate billions of tokens in RAM.
# ============================================================

import os
import math
import numpy as np
import tiktoken

from datasets import load_dataset


# ============================================================
# CONFIGURATION
# ============================================================

# DATA_DIR must already exist in your training script.
# Example:
# DATA_DIR = "/content/drive/MyDrive/gpt_data"

if "DATA_DIR" not in globals():
    raise RuntimeError(
        "DATA_DIR must be defined before running this data-preparation block."
    )

os.makedirs(DATA_DIR, exist_ok=True)

TRAIN_DIR = os.path.join(DATA_DIR, "pretrain_train_shards")
VAL_BIN = os.path.join(DATA_DIR, "pretrain_val.bin")

# Total target training tokens.
TARGET_TRAIN_TOKENS = 5_000_000_000

# Dataset mixture.
FINEWEB_EDU_RATIO = 0.70
FINEWEB_RATIO = 0.30

# Binary shard size.
#
# 10M uint16 tokens ~= 20 MB.
# This is deliberately small enough to keep RAM usage modest.
TOKENS_PER_SHARD = 10_000_000

# Validation size.
VAL_TOKENS = 10_000_000

# GPT-2 BPE.
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token

VOCAB_SIZE = enc.n_vocab

# Reproducibility.
SEED = 1337

# Dataset block size used for reporting / sanity checks.
BLOCK_SIZE = 1024


# ============================================================
# VALIDATE CONFIG
# ============================================================

if not math.isclose(
    FINEWEB_EDU_RATIO + FINEWEB_RATIO,
    1.0,
    rel_tol=0.0,
    abs_tol=1e-9,
):
    raise ValueError(
        "FINEWEB_EDU_RATIO + FINEWEB_RATIO must equal 1.0"
    )

if TARGET_TRAIN_TOKENS <= 0:
    raise ValueError("TARGET_TRAIN_TOKENS must be > 0")

if TOKENS_PER_SHARD <= 0:
    raise ValueError("TOKENS_PER_SHARD must be > 0")

if VAL_TOKENS <= 0:
    raise ValueError("VAL_TOKENS must be > 0")

if VOCAB_SIZE != 50257:
    raise ValueError(
        f"Unexpected GPT-2 vocabulary size: {VOCAB_SIZE}"
    )

print("=" * 70)
print("V4 PRETRAINING DATA CONFIGURATION")
print("=" * 70)

print(f"GPT-2 vocabulary       : {VOCAB_SIZE:,}")
print(f"Target train tokens    : {TARGET_TRAIN_TOKENS:,}")
print(f"FineWeb-Edu ratio      : {FINEWEB_EDU_RATIO:.0%}")
print(f"FineWeb ratio          : {FINEWEB_RATIO:.0%}")
print(f"Tokens per shard       : {TOKENS_PER_SHARD:,}")
print(f"Validation tokens      : {VAL_TOKENS:,}")
print(f"Output directory       : {TRAIN_DIR}")
print()


# ============================================================
# TOKENIZER
# ============================================================

def tokenize_text(text):
    """
    Tokenize one document with GPT-2 BPE and append EOT.

    GPT-2's encode_ordinary() is intentionally used so special
    token strings appearing inside the source text are not
    interpreted as special tokens.
    """

    if not text:
        return None

    text = text.strip()

    if not text:
        return None

    tokens = enc.encode_ordinary(text)
    tokens.append(EOT)

    return np.asarray(tokens, dtype=np.uint16)


# ============================================================
# LOAD STREAMING DATASETS
# ============================================================

print("=" * 70)
print("LOADING STREAMING DATASETS")
print("=" * 70)

# FineWeb-Edu provides sample-10BT.
fineweb_edu = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True,
)

# FineWeb provides sample-10BT.
fineweb = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train",
    streaming=True,
)

print("FineWeb-Edu streaming dataset : READY")
print("FineWeb streaming dataset     : READY")
print()


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(TRAIN_DIR, exist_ok=True)


# ============================================================
# OPTIONAL RESUME SUPPORT
# ============================================================

# Delete this block if you always want to regenerate the corpus.
#
# Existing shards are detected, but the streaming iterators cannot
# automatically resume from the exact document position. Therefore,
# for a deterministic restart, remove old shards before rebuilding.

existing_shards = sorted(
    f
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".bin")
)

if existing_shards:
    print(
        f"Found {len(existing_shards)} existing training shards."
    )
    print(
        "Delete them manually if you want to rebuild the dataset."
    )
    print()


# ============================================================
# STREAM ITERATORS
# ============================================================

edu_iter = iter(fineweb_edu)
web_iter = iter(fineweb)


# ============================================================
# DOCUMENT TOKEN STREAM
# ============================================================

def next_document_tokens(dataset_iter):
    """
    Return the next non-empty tokenized document.

    Returns:
        np.ndarray(dtype=np.uint16)
        or None if the stream ends.
    """

    while True:
        try:
            example = next(dataset_iter)
        except StopIteration:
            return None

        text = example.get("text", None)

        tokens = tokenize_text(text)

        if tokens is not None and len(tokens) > 0:
            return tokens


# ============================================================
# WRITE ONE SOURCE CHUNK DIRECTLY TO DISK
# ============================================================

def write_source_tokens(
    dataset_iter,
    file_handle,
    target_tokens,
):
    """
    Stream documents from one dataset and write approximately
    target_tokens directly into the current shard.

    Returns:
        number of tokens actually written
    """

    written = 0

    while written < target_tokens:

        tokens = next_document_tokens(dataset_iter)

        if tokens is None:
            break

        remaining = target_tokens - written

        if len(tokens) <= remaining:
            tokens_to_write = tokens
        else:
            # Preserve a document boundary when possible.
            #
            # If the remaining space is at least 2 tokens, reserve
            # one final position for EOT.
            if remaining >= 2:
                tokens_to_write = np.concatenate(
                    [
                        tokens[:remaining - 1],
                        np.asarray([EOT], dtype=np.uint16),
                    ]
                )
            else:
                tokens_to_write = np.asarray(
                    [EOT],
                    dtype=np.uint16,
                )

        file_handle.write(tokens_to_write.tobytes())

        written += len(tokens_to_write)

        if written >= target_tokens:
            break

    return written


# ============================================================
# CLEAN OLD SHARDS
# ============================================================

# For a fresh dataset build, remove old shards.
#
# Set REBUILD_DATA = False to keep existing files.

REBUILD_DATA = True

if REBUILD_DATA:
    old_files = [
        f for f in os.listdir(TRAIN_DIR)
        if f.endswith(".bin")
    ]

    for filename in old_files:
        os.remove(
            os.path.join(TRAIN_DIR, filename)
        )

    print(
        f"Removed {len(old_files)} old training shards."
    )


# ============================================================
# TRAINING SHARD GENERATION
# ============================================================

print()
print("=" * 70)
print("CREATING TRAINING SHARDS")
print("=" * 70)

edu_total_target = int(
    TARGET_TRAIN_TOKENS * FINEWEB_EDU_RATIO
)

web_total_target = int(
    TARGET_TRAIN_TOKENS * FINEWEB_RATIO
)

print(
    f"FineWeb-Edu target : {edu_total_target:,}"
)

print(
    f"FineWeb target     : {web_total_target:,}"
)

print(
    f"Total target       : "
    f"{edu_total_target + web_total_target:,}"
)

print()


# To obtain an approximately exact 70/30 mixture in every full shard:
#
#   7,000,000 FineWeb-Edu
#   3,000,000 FineWeb
#
# for each 10M-token shard.
#
# The source sections are written consecutively within each shard,
# while shard order is randomized later by the training loader.

edu_per_shard = int(
    TOKENS_PER_SHARD * FINEWEB_EDU_RATIO
)

web_per_shard = (
    TOKENS_PER_SHARD - edu_per_shard
)

edu_written_total = 0
web_written_total = 0

shard_index = 0

while (
    edu_written_total < edu_total_target
    or web_written_total < web_total_target
):

    shard_path = os.path.join(
        TRAIN_DIR,
        f"train_{shard_index:05d}.bin"
    )

    # Remaining targets.
    edu_remaining = (
        edu_total_target - edu_written_total
    )

    web_remaining = (
        web_total_target - web_written_total
    )

    # Full shard uses 70/30.
    #
    # Final shard may be smaller if one source reaches its target.
    edu_target = min(
        edu_per_shard,
        max(0, edu_remaining),
    )

    web_target = min(
        web_per_shard,
        max(0, web_remaining),
    )

    # If one source is exhausted, use remaining capacity from the
    # other source so the requested total token target is reached.
    current_target = edu_target + web_target

    if current_target < TOKENS_PER_SHARD:

        remaining_capacity = (
            TOKENS_PER_SHARD - current_target
        )

        if edu_remaining > edu_target:
            extra = min(
                remaining_capacity,
                edu_remaining - edu_target,
            )
            edu_target += extra
            remaining_capacity -= extra

        if (
            remaining_capacity > 0
            and web_remaining > web_target
        ):
            extra = min(
                remaining_capacity,
                web_remaining - web_target,
            )
            web_target += extra

    with open(
        shard_path,
        "wb",
        buffering=1024 * 1024,
    ) as f:

        # ----------------------------------------------------
        # FineWeb-Edu section
        # ----------------------------------------------------

        edu_written = write_source_tokens(
            edu_iter,
            f,
            edu_target,
        )

        # ----------------------------------------------------
        # FineWeb section
        # ----------------------------------------------------

        web_written = write_source_tokens(
            web_iter,
            f,
            web_target,
        )

    edu_written_total += edu_written
    web_written_total += web_written

    shard_tokens = (
        edu_written + web_written
    )

    # Empty shard safety.
    if shard_tokens == 0:
        os.remove(shard_path)

        raise RuntimeError(
            "Both streaming datasets ended before "
            "the requested token target was reached."
        )

    print(
        f"Shard {shard_index:05d} | "
        f"tokens={shard_tokens:,} | "
        f"Edu={edu_written:,} | "
        f"FineWeb={web_written:,}"
    )

    shard_index += 1

    # If either stream ended prematurely, the next loop would
    # repeatedly produce zero tokens. Stop clearly.
    if (
        edu_written == 0
        and edu_remaining > 0
        and edu_target > 0
    ):
        print(
            "WARNING: FineWeb-Edu stream ended early."
        )
        break

    if (
        web_written == 0
        and web_remaining > 0
        and web_target > 0
    ):
        print(
            "WARNING: FineWeb stream ended early."
        )
        break


# ============================================================
# SHARD MANIFEST
# ============================================================

train_shards = sorted(
    os.path.join(TRAIN_DIR, f)
    for f in os.listdir(TRAIN_DIR)
    if f.endswith(".bin")
)

rng = np.random.default_rng(SEED)

rng.shuffle(train_shards)

manifest_path = os.path.join(
    TRAIN_DIR,
    "shards.txt"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    for path in train_shards:
        f.write(
            os.path.basename(path) + "\n"
        )


# ============================================================
# TRAINING SUMMARY
# ============================================================

actual_train_tokens = (
    edu_written_total + web_written_total
)

print()
print("=" * 70)
print("TRAINING DATA COMPLETE")
print("=" * 70)

print(
    f"FineWeb-Edu tokens written : "
    f"{edu_written_total:,}"
)

print(
    f"FineWeb tokens written     : "
    f"{web_written_total:,}"
)

print(
    f"Total tokens written       : "
    f"{actual_train_tokens:,}"
)

if actual_train_tokens > 0:
    actual_edu_ratio = (
        edu_written_total / actual_train_tokens
    )

    actual_web_ratio = (
        web_written_total / actual_train_tokens
    )

    print(
        f"Actual Edu ratio           : "
        f"{actual_edu_ratio:.4%}"
    )

    print(
        f"Actual FineWeb ratio       : "
        f"{actual_web_ratio:.4%}"
    )

print(
    f"Training shards             : "
    f"{len(train_shards):,}"
)

print(
    f"Shard size target           : "
    f"{TOKENS_PER_SHARD:,} tokens"
)

print(
    f"Manifest                    : "
    f"{manifest_path}"
)


# ============================================================
# VALIDATION DATA
# WikiText-103 validation is intentionally separate from the
# FineWeb/FineWeb-Edu training corpus.
# ============================================================

print()
print("=" * 70)
print("CREATING VALIDATION DATA")
print("=" * 70)

wikitext = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1",
    split="validation",
    streaming=True,
)

val_iter = iter(wikitext)

val_written = 0

with open(
    VAL_BIN,
    "wb",
    buffering=1024 * 1024,
) as f:

    while val_written < VAL_TOKENS:

        tokens = next_document_tokens(
            val_iter
        )

        if tokens is None:
            break

        remaining = (
            VAL_TOKENS - val_written
        )

        if len(tokens) <= remaining:
            tokens_to_write = tokens
        else:
            if remaining >= 2:
                tokens_to_write = np.concatenate(
                    [
                        tokens[:remaining - 1],
                        np.asarray(
                            [EOT],
                            dtype=np.uint16,
                        ),
                    ]
                )
            else:
                tokens_to_write = np.asarray(
                    [EOT],
                    dtype=np.uint16,
                )

        f.write(
            tokens_to_write.tobytes()
        )

        val_written += len(
            tokens_to_write
        )

        if (
            val_written % 1_000_000
            < len(tokens_to_write)
        ):
            print(
                f"Validation tokens: "
                f"{val_written:,} / "
                f"{VAL_TOKENS:,}"
            )

print()
print(
    f"Validation tokens written: "
    f"{val_written:,}"
)


# ============================================================
# FINAL FILE SUMMARY
# ============================================================

print()
print("=" * 70)
print("V4 PRETRAINING DATA READY")
print("=" * 70)

print(
    f"Train directory : {TRAIN_DIR}"
)

print(
    f"Train shards    : {len(train_shards):,}"
)

print(
    f"Train tokens    : {actual_train_tokens:,}"
)

print(
    f"Validation file : {VAL_BIN}"
)

print(
    f"Validation tokens: {val_written:,}"
)

print()

train_bytes = sum(
    os.path.getsize(path)
    for path in train_shards
)

val_bytes = (
    os.path.getsize(VAL_BIN)
    if os.path.exists(VAL_BIN)
    else 0
)

print(
    f"Train disk size : "
    f"{train_bytes / (1024**3):.2f} GB"
)

print(
    f"Val disk size   : "
    f"{val_bytes / (1024**2):.2f} MB"
)

print()
print("Training shards:")

for path in train_shards[:10]:
    print(
        " ",
        os.path.basename(path),
        f"{os.path.getsize(path) / (1024**2):.1f} MB",
    )

if len(train_shards) > 10:
    print(
        f"  ... {len(train_shards) - 10:,} more shards"
    )

print()
print("✅ Dataset preparation complete.")

c:\Users\sandyarjun\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


V4 PRETRAINING DATA CONFIGURATION
GPT-2 vocabulary       : 50,257
Target train tokens    : 5,000,000,000
FineWeb-Edu ratio      : 70%
FineWeb ratio          : 30%
Tokens per shard       : 10,000,000
Validation tokens      : 10,000,000
Output directory       : C:\Users\sandyarjun\gpt150m_project\data\pretrain_train_shards

LOADING STREAMING DATASETS


FineWeb-Edu streaming dataset : READY
FineWeb streaming dataset     : READY

Found 500 existing training shards.
Delete them manually if you want to rebuild the dataset.

Removed 500 old training shards.

CREATING TRAINING SHARDS
FineWeb-Edu target : 3,500,000,000
FineWeb target     : 1,500,000,000
Total target       : 5,000,000,000

Shard 00000 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00001 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00002 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00003 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00004 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00005 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00006 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00007 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00008 | tokens=10,000,000 | Edu=7,000,000 | FineWeb=3,000,000
Shard 00009 | tokens=10,000,000 | Edu=7,000,000 | Fi

## 5. Pretraining loop

Loads the FineWeb-Edu/FineWeb training shards and WikiText-103 validation tokens, then trains the model for up to 152,588 optimizer steps, or about 5 billion tokens at 32,768 tokens per step. It evaluates every 1,000 steps, saves `latest.pt` every 1,000 steps, saves `best.pt` when validation loss improves, and resumes automatically from `latest.pt`.

In [5]:
import os
import numpy as np

TRAIN_SHARD_DIR = os.path.join(
    DATA_DIR,
    "pretrain_train_shards"
)

train_shards = sorted(
    os.path.join(TRAIN_SHARD_DIR, f)
    for f in os.listdir(TRAIN_SHARD_DIR)
    if f.endswith(".bin")
)

if not train_shards:
    raise FileNotFoundError(
        f"No training shards found in:\n{TRAIN_SHARD_DIR}"
    )

total_tokens = 0

for shard_path in train_shards:
    data = np.memmap(
        shard_path,
        dtype=np.uint16,
        mode="r"
    )

    total_tokens += len(data)

print(f"Training shards : {len(train_shards):,}")
print(f"Total tokens    : {total_tokens:,}")
print(f"Total tokens    : {total_tokens / 1e9:.3f}B")


Training shards : 500
Total tokens    : 5,000,000,000
Total tokens    : 5.000B


In [6]:
# ============================================================
# V4 PRETRAINING LOOP
# 166M PARAMETER MoE MODEL
#
# Architecture:
#   - RMSNorm
#   - GQA: 8 Q heads / 4 KV heads
#   - QK-Norm
#   - NeoX-style RoPE
#   - SDPA / FlashAttention backend
#   - Top-2 / 4-expert SwiGLU MoE
#   - MoE auxiliary load-balancing loss
#   - Weight tying
#   - Gradient checkpointing
#
# Training:
#   - FineWeb-Edu + FineWeb sharded uint16 dataset
#   - 1024-token sequences
#   - BF16 / FP16
#   - Gradient accumulation
#   - AdamW
#   - Gradient clipping
#   - Resume support
#   - Best/latest checkpoints
#   - Validation loss
#   - LM loss + MoE aux loss logging
#   - Peak VRAM monitoring
#   - Optional torch.compile
# ============================================================

import math
import time
import os
import inspect
import random

import numpy as np
import torch


# ============================================================
# REQUIRED GLOBALS
# ============================================================

if "DATA_DIR" not in globals():
    raise RuntimeError(
        "DATA_DIR must be defined before starting training."
    )

if "GPT" not in globals():
    raise RuntimeError(
        "GPT class must be defined before starting training."
    )

if "GPTConfig" not in globals():
    raise RuntimeError(
        "GPTConfig must be defined before starting training."
    )


# ============================================================
# PATHS
# ============================================================

PRETRAIN_TRAIN_SHARD_DIR = os.path.join(
    DATA_DIR,
    "pretrain_train_shards",
)

PRETRAIN_VAL_BIN = os.path.join(
    DATA_DIR,
    "pretrain_val.bin",
)

PRETRAIN_CKPT_DIR = os.path.join(
    DATA_DIR,
    "pretrain_checkpoints",
)

os.makedirs(
    PRETRAIN_CKPT_DIR,
    exist_ok=True,
)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================
# ============================================================
# FULL 5B-TOKEN RUN
# ============================================================

PRETRAIN_BLOCK_SIZE = 1024

PRETRAIN_BATCH_SIZE = 1
PRETRAIN_GRAD_ACCUM = 32

# 32,768 tokens / optimizer step
TOKENS_PER_STEP = (
    PRETRAIN_BATCH_SIZE
    * PRETRAIN_GRAD_ACCUM
    * PRETRAIN_BLOCK_SIZE
)

# ~5B tokens
PRETRAIN_MAX_STEPS = 152_588

# Warmup
PRETRAIN_WARMUP_STEPS = 2_000

# LR
PRETRAIN_MAX_LR = 3e-5
PRETRAIN_MIN_LR = 3e-6

# AdamW
PRETRAIN_WEIGHT_DECAY = 0.1

# Gradient clipping
PRETRAIN_GRAD_CLIP = 1.0

# Evaluation/checkpointing
PRETRAIN_EVAL_INTERVAL = 1_000
PRETRAIN_CKPT_INTERVAL = 1_000
PRETRAIN_EVAL_ITERS = 50

# Keep these off initially
USE_TORCH_COMPILE = False
CHECKPOINT_DEBUG = False

SEED = 1337


# ============================================================
# SEED
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# PRECISION
# ============================================================

if (
    device == "cuda"
    and torch.cuda.is_bf16_supported()
):
    ptdtype = torch.bfloat16
    use_grad_scaler = False

elif device == "cuda":
    ptdtype = torch.float16
    use_grad_scaler = True

else:
    ptdtype = torch.float32
    use_grad_scaler = False


# ============================================================
# CUDA SETTINGS
# ============================================================

if device == "cuda":

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    torch.set_float32_matmul_precision("high")


# ============================================================
# GPU PEAK FLOPS
# MFU diagnostic only.
# ============================================================

GPU_PEAK_FLOPS = {
    "RTX 5050": 26e12,
    "T4": 65e12,
    "A100": 312e12,
}

if device == "cuda":

    PEAK_FLOPS = 30e12

    gpu_name = torch.cuda.get_device_name(0)

    for key, value in GPU_PEAK_FLOPS.items():

        if key in gpu_name:
            PEAK_FLOPS = value
            break

else:

    PEAK_FLOPS = 1e12
    gpu_name = "CPU"


# ============================================================
# BOOTSTRAP / VALIDATE CONFIG
# ============================================================

if "cfg" not in globals():

    cfg = GPTConfig(
        block_size=PRETRAIN_BLOCK_SIZE,
        vocab_size=50257,

        n_layer=12,
        n_head=8,
        n_kv_head=4,
        n_embd=864,

        dropout=0.0,
        bias=False,

        rope_theta=10000.0,
        rope_scaling_factor=1.0,

        qk_norm=True,

        moe_num_experts=4,
        moe_top_k=2,
        moe_hidden_dim=768,
        moe_aux_loss_coeff=0.01,

        gradient_checkpointing=True,
    )

else:

    if cfg.block_size != PRETRAIN_BLOCK_SIZE:
        raise ValueError(
            f"Config block_size={cfg.block_size} "
            f"but PRETRAIN_BLOCK_SIZE={PRETRAIN_BLOCK_SIZE}"
        )


# ============================================================
# CONFIG SANITY CHECK
# ============================================================

expected = {
    "block_size": PRETRAIN_BLOCK_SIZE,
    "vocab_size": 50257,
    "n_layer": 12,
    "n_head": 8,
    "n_kv_head": 4,
    "n_embd": 864,
    "moe_num_experts": 4,
    "moe_top_k": 2,
    "moe_hidden_dim": 768,
}

for name, expected_value in expected.items():

    actual_value = getattr(
        cfg,
        name,
    )

    if actual_value != expected_value:

        raise ValueError(
            f"V4 configuration mismatch: "
            f"{name}={actual_value}, "
            f"expected {expected_value}"
        )


# ============================================================
# HARDWARE / TRAINING SUMMARY
# ============================================================

tokens_per_optimizer_step = (
    PRETRAIN_BATCH_SIZE
    * PRETRAIN_GRAD_ACCUM
    * PRETRAIN_BLOCK_SIZE
)

print()
print("=" * 70)
print("V4 PRETRAINING SETUP")
print("=" * 70)

print(
    f"device                  : {device}"
)

print(
    f"GPU                     : {gpu_name}"
)

print(
    f"dtype                   : {ptdtype}"
)

print(
    f"GradScaler              : {use_grad_scaler}"
)

print(
    f"micro batch             : {PRETRAIN_BATCH_SIZE}"
)

print(
    f"grad accumulation       : {PRETRAIN_GRAD_ACCUM}"
)

print(
    f"effective batch         : "
    f"{PRETRAIN_BATCH_SIZE * PRETRAIN_GRAD_ACCUM}"
)

print(
    f"sequence length         : "
    f"{PRETRAIN_BLOCK_SIZE}"
)

print(
    f"tokens/update           : "
    f"{tokens_per_optimizer_step:,}"
)

print(
    f"max steps               : "
    f"{PRETRAIN_MAX_STEPS:,}"
)

print(
    f"max training tokens     : "
    f"{tokens_per_optimizer_step * PRETRAIN_MAX_STEPS:,}"
)

print(
    f"torch.compile           : "
    f"{USE_TORCH_COMPILE}"
)

print(
    f"checkpoint debug        : "
    f"{CHECKPOINT_DEBUG}"
)

print("=" * 70)


# ============================================================
# FIND TRAINING SHARDS
# ============================================================

if not os.path.isdir(
    PRETRAIN_TRAIN_SHARD_DIR
):

    raise FileNotFoundError(
        "Training shard directory not found:\n"
        f"{PRETRAIN_TRAIN_SHARD_DIR}"
    )


TRAIN_SHARD_PATHS = sorted(
    os.path.join(
        PRETRAIN_TRAIN_SHARD_DIR,
        filename,
    )
    for filename in os.listdir(
        PRETRAIN_TRAIN_SHARD_DIR
    )
    if filename.endswith(".bin")
)


if not TRAIN_SHARD_PATHS:

    raise FileNotFoundError(
        "No training shards found:\n"
        f"{PRETRAIN_TRAIN_SHARD_DIR}"
    )


# ============================================================
# LOAD TRAIN MEMMAPS
# ============================================================

print()
print("=" * 70)
print("LOADING TRAINING SHARDS")
print("=" * 70)

TRAIN_MEMMAPS = []

total_train_tokens = 0


for path in TRAIN_SHARD_PATHS:

    data = np.memmap(
        path,
        dtype=np.uint16,
        mode="r",
    )

    if len(data) <= (
        PRETRAIN_BLOCK_SIZE + 1
    ):

        raise ValueError(
            f"Shard is too small:\n"
            f"{path}\n"
            f"tokens={len(data)}"
        )

    TRAIN_MEMMAPS.append(data)

    total_train_tokens += len(data)


TRAIN_SHARD_LENGTHS = np.asarray(
    [
        len(data)
        for data in TRAIN_MEMMAPS
    ],
    dtype=np.float64,
)


TRAIN_SHARD_PROBS = (
    TRAIN_SHARD_LENGTHS
    / TRAIN_SHARD_LENGTHS.sum()
)


print(
    f"training shards       : "
    f"{len(TRAIN_MEMMAPS):,}"
)

print(
    f"training tokens       : "
    f"{total_train_tokens:,}"
)

print(
    f"training tokens       : "
    f"{total_train_tokens / 1e9:.3f}B"
)

print("=" * 70)


# ============================================================
# VALIDATION MEMMAP
# ============================================================

if not os.path.exists(
    PRETRAIN_VAL_BIN
):

    raise FileNotFoundError(
        "Validation file not found:\n"
        f"{PRETRAIN_VAL_BIN}"
    )


VAL_DATA = np.memmap(
    PRETRAIN_VAL_BIN,
    dtype=np.uint16,
    mode="r",
)


if len(VAL_DATA) <= (
    PRETRAIN_BLOCK_SIZE + 1
):

    raise ValueError(
        "Validation dataset is too small."
    )


print(
    f"validation tokens    : "
    f"{len(VAL_DATA):,}"
)


# ============================================================
# DATA RNG
# ============================================================

DATA_RNG = np.random.default_rng(
    SEED
)


# ============================================================
# BATCH LOADER
# ============================================================

def get_batch(
    split,
    block_size,
    batch_size,
    device,
):
    """
    Sample random next-token-prediction sequences.

    x:
        tokens[t : t + block_size]

    y:
        tokens[t + 1 : t + block_size + 1]
    """

    if split not in (
        "train",
        "val",
    ):

        raise ValueError(
            "split must be 'train' or 'val'"
        )


    x_np = np.empty(
        (
            batch_size,
            block_size,
        ),
        dtype=np.int64,
    )

    y_np = np.empty_like(x_np)


    # ========================================================
    # TRAIN
    # ========================================================

    if split == "train":

        for b in range(batch_size):

            shard_idx = (
                DATA_RNG.choice(
                    len(TRAIN_MEMMAPS),
                    p=TRAIN_SHARD_PROBS,
                )
            )

            data = TRAIN_MEMMAPS[
                shard_idx
            ]

            max_start = (
                len(data)
                - block_size
                - 1
            )

            if max_start <= 0:

                raise RuntimeError(
                    "Training shard too small."
                )

            start = DATA_RNG.integers(
                0,
                max_start + 1,
            )

            chunk = data[
                start:
                start + block_size + 1
            ]

            x_np[b] = chunk[:-1]
            y_np[b] = chunk[1:]


    # ========================================================
    # VALIDATION
    # ========================================================

    else:

        max_start = (
            len(VAL_DATA)
            - block_size
            - 1
        )

        if max_start <= 0:

            raise RuntimeError(
                "Validation dataset too small."
            )

        starts = DATA_RNG.integers(
            0,
            max_start + 1,
            size=batch_size,
        )

        for b, start in enumerate(
            starts
        ):

            chunk = VAL_DATA[
                start:
                start + block_size + 1
            ]

            x_np[b] = chunk[:-1]
            y_np[b] = chunk[1:]


    # ========================================================
    # TORCH
    # ========================================================

    x = torch.from_numpy(x_np)
    y = torch.from_numpy(y_np)


    # ========================================================
    # DEVICE
    # ========================================================

    if device == "cuda":

        x = x.pin_memory().to(
            device,
            non_blocking=True,
        )

        y = y.pin_memory().to(
            device,
            non_blocking=True,
        )

    else:

        x = x.to(device)
        y = y.to(device)


    return x, y


# ============================================================
# TEST DATA LOADER
# ============================================================

print()
print("Testing data loader...")


test_x, test_y = get_batch(
    "train",
    PRETRAIN_BLOCK_SIZE,
    PRETRAIN_BATCH_SIZE,
    device,
)


if (
    tuple(test_x.shape)
    != (
        PRETRAIN_BATCH_SIZE,
        PRETRAIN_BLOCK_SIZE,
    )
):

    raise RuntimeError(
        f"Unexpected training shape: "
        f"{test_x.shape}"
    )


if (
    test_x.min().item() < 0
    or test_x.max().item() >= cfg.vocab_size
):

    raise RuntimeError(
        "Invalid training token ID."
    )


val_test_x, val_test_y = get_batch(
    "val",
    PRETRAIN_BLOCK_SIZE,
    PRETRAIN_BATCH_SIZE,
    device,
)


if (
    tuple(val_test_x.shape)
    != (
        PRETRAIN_BATCH_SIZE,
        PRETRAIN_BLOCK_SIZE,
    )
):

    raise RuntimeError(
        f"Unexpected validation shape: "
        f"{val_test_x.shape}"
    )


if (
    val_test_x.min().item() < 0
    or val_test_x.max().item() >= cfg.vocab_size
):

    raise RuntimeError(
        "Invalid validation token ID."
    )


print(
    f"train shape  : {tuple(test_x.shape)}"
)

print(
    f"train dtype  : {test_x.dtype}"
)

print(
    f"val shape    : {tuple(val_test_x.shape)}"
)

print(
    "✅ Data loader sanity check PASSED"
)


del test_x
del test_y
del val_test_x
del val_test_y


# ============================================================
# CREATE MODEL
# ============================================================

print()
print("=" * 70)
print("CREATING V4 MODEL")
print("=" * 70)

model = GPT(cfg).to(device)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

non_embedding_params = (
    model.get_num_params()
)


print(
    f"total parameters      : "
    f"{total_params / 1e6:.2f}M"
)

print(
    f"non-embedding params  : "
    f"{non_embedding_params / 1e6:.2f}M"
)

print("=" * 70)


# ============================================================
# CHECKPOINT DEBUG
# ============================================================

if CHECKPOINT_DEBUG:

    try:

        torch.utils.checkpoint.set_checkpoint_debug_enabled(
            True
        )

        print(
            "checkpoint debug     : ENABLED"
        )

    except Exception:

        print(
            "checkpoint debug     : unavailable"
        )


# ============================================================
# OPTIMIZER
# ============================================================

def build_optimizer(
    model,
    learning_rate,
    weight_decay,
    betas,
    device,
):
    """
    AdamW.

    2D+ parameters:
        weight decay

    1D parameters:
        no weight decay
    """

    decay_params = []
    no_decay_params = []


    for name, parameter in (
        model.named_parameters()
    ):

        if not parameter.requires_grad:
            continue

        if parameter.dim() >= 2:

            decay_params.append(
                parameter
            )

        else:

            no_decay_params.append(
                parameter
            )


    optimizer_groups = [
        {
            "params": decay_params,
            "weight_decay": weight_decay,
        },
        {
            "params": no_decay_params,
            "weight_decay": 0.0,
        },
    ]


    optimizer_kwargs = {
        "lr": learning_rate,
        "betas": betas,
        "eps": 1e-8,
    }


    fused_supported = (
        "fused"
        in inspect.signature(
            torch.optim.AdamW
        ).parameters
    )


    if (
        device == "cuda"
        and fused_supported
    ):

        optimizer_kwargs[
            "fused"
        ] = True

        print(
            "AdamW fused           : True"
        )

    else:

        print(
            "AdamW fused           : False"
        )


    return torch.optim.AdamW(
        optimizer_groups,
        **optimizer_kwargs,
    )


optimizer = build_optimizer(
    model=model,
    learning_rate=PRETRAIN_MAX_LR,
    weight_decay=PRETRAIN_WEIGHT_DECAY,
    betas=(0.9, 0.95),
    device=device,
)


# ============================================================
# GRAD SCALER
# ============================================================

if use_grad_scaler:

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

else:

    scaler = None


# ============================================================
# CHECKPOINT PATHS
# ============================================================

pretrain_ckpt_path = os.path.join(
    PRETRAIN_CKPT_DIR,
    "latest.pt",
)

pretrain_best_ckpt_path = os.path.join(
    PRETRAIN_CKPT_DIR,
    "best.pt",
)


# ============================================================
# RESUME
# ============================================================

start_step = 0
best_val_loss = float("inf")


if os.path.exists(
    pretrain_ckpt_path
):

    print()
    print("=" * 70)
    print("RESUMING FROM CHECKPOINT")
    print("=" * 70)

    print(
        pretrain_ckpt_path
    )


    try:

        ckpt = torch.load(
            pretrain_ckpt_path,
            map_location=device,
            weights_only=False,
        )

    except TypeError:

        ckpt = torch.load(
            pretrain_ckpt_path,
            map_location=device,
        )


    # --------------------------------------------------------
    # CHECK CONFIG
    # --------------------------------------------------------

    saved_cfg = ckpt.get(
        "config",
        None,
    )


    if saved_cfg is not None:

        fields_to_check = [
            "block_size",
            "vocab_size",
            "n_layer",
            "n_head",
            "n_kv_head",
            "n_embd",
            "moe_num_experts",
            "moe_top_k",
            "moe_hidden_dim",
        ]


        for field in fields_to_check:

            current_value = getattr(
                cfg,
                field,
            )

            saved_value = getattr(
                saved_cfg,
                field,
            )

            if (
                current_value
                != saved_value
            ):

                raise ValueError(
                    f"Checkpoint configuration "
                    f"mismatch for '{field}': "
                    f"current={current_value}, "
                    f"saved={saved_value}"
                )


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    model.load_state_dict(
        ckpt["model"]
    )


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )


    # --------------------------------------------------------
    # SCALER
    # --------------------------------------------------------

    if (
        scaler is not None
        and ckpt.get("scaler") is not None
    ):

        scaler.load_state_dict(
            ckpt["scaler"]
        )


    # --------------------------------------------------------
    # STEP
    # --------------------------------------------------------

    start_step = (
        ckpt["step"] + 1
    )


    best_val_loss = ckpt.get(
        "best_val_loss",
        float("inf"),
    )


    # --------------------------------------------------------
    # DATA RNG
    # --------------------------------------------------------

    if (
        "data_rng_state"
        in ckpt
    ):

        DATA_RNG.bit_generator.state = (
            ckpt[
                "data_rng_state"
            ]
        )


    # --------------------------------------------------------
    # CPU RNG
    # --------------------------------------------------------

    if (
        "torch_rng_state"
        in ckpt
    ):

        torch.set_rng_state(
            ckpt[
                "torch_rng_state"
            ]
        )


    # --------------------------------------------------------
    # PYTHON RNG
    # --------------------------------------------------------

    if (
        "python_rng_state"
        in ckpt
    ):

        random.setstate(
            ckpt[
                "python_rng_state"
            ]
        )


    # --------------------------------------------------------
    # CUDA RNG
    # --------------------------------------------------------

    if (
        device == "cuda"
        and
        "cuda_rng_state_all"
        in ckpt
    ):

        torch.cuda.set_rng_state_all(
            ckpt[
                "cuda_rng_state_all"
            ]
        )


    print(
        f"resumed step          : "
        f"{start_step}"
    )

    print(
        f"best validation loss  : "
        f"{best_val_loss:.4f}"
    )


else:

    print(
        "No checkpoint found."
    )

    print(
        "Starting from scratch."
    )


# ============================================================
# TORCH COMPILE
# ============================================================

compiled_model = model


if (
    device == "cuda"
    and USE_TORCH_COMPILE
):

    try:

        print(
            "Initializing torch.compile..."
        )

        compiled_model = torch.compile(
            model
        )

        print(
            "torch.compile enabled."
        )

    except Exception as e:

        print(
            "torch.compile failed:"
        )

        print(e)

        print(
            "Using eager mode."
        )

        compiled_model = model


# ============================================================
# LEARNING-RATE SCHEDULE
# ============================================================

def get_lr(
    step,
    warmup_steps,
    max_steps,
    max_lr,
    min_lr,
):

    if step < warmup_steps:

        return (
            max_lr
            * (step + 1)
            / warmup_steps
        )


    if step >= max_steps:

        return min_lr


    decay_ratio = (
        step - warmup_steps
    ) / (
        max_steps - warmup_steps
    )


    coefficient = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * decay_ratio
            )
        )
    )


    return (
        min_lr
        + coefficient
        * (
            max_lr
            - min_lr
        )
    )


# ============================================================
# FORWARD HELPER
# ============================================================

def forward_train(
    model,
    x,
    y,
):
    """
    Current V4 GPT.forward():

        use_cache=False
        ->
        logits, total_loss, total_aux_loss

    The returned total loss is:

        total_loss =
            lm_loss
            + moe_aux_loss_coeff * total_aux_loss

    We recover lm_loss algebraically without running
    another cross-entropy operation.
    """

    result = model(
        x,
        targets=y,
        use_cache=False,
    )


    if len(result) != 3:

        raise RuntimeError(
            "Expected V4 GPT.forward() to return "
            "(logits, loss, total_aux_loss). "
            f"Got {len(result)} outputs."
        )


    logits = result[0]
    total_loss = result[1]
    aux_loss = result[2]


    if aux_loss is None:

        lm_loss = total_loss

    else:

        lm_loss = (
            total_loss
            - (
                cfg.moe_aux_loss_coeff
                * aux_loss
            )
        )


    return (
        logits,
        total_loss,
        lm_loss,
        aux_loss,
    )


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def estimate_loss(
    model,
    block_size,
    batch_size,
    device,
    ptdtype,
    eval_iters=25,
):

    was_training = model.training

    model.eval()


    result = {}


    for split in (
        "train",
        "val",
    ):

        total_loss_sum = 0.0
        lm_loss_sum = 0.0
        aux_loss_sum = 0.0


        for _ in range(
            eval_iters
        ):

            x, y = get_batch(
                split,
                block_size,
                batch_size,
                device,
            )


            if device == "cuda":

                with torch.autocast(
                    device_type="cuda",
                    dtype=ptdtype,
                ):

                    (
                        _,
                        total_loss,
                        lm_loss,
                        aux_loss,
                    ) = forward_train(
                        model,
                        x,
                        y,
                    )

            else:

                (
                    _,
                    total_loss,
                    lm_loss,
                    aux_loss,
                ) = forward_train(
                    model,
                    x,
                    y,
                )


            total_loss_sum += (
                total_loss
                .detach()
                .float()
                .item()
            )


            lm_loss_sum += (
                lm_loss
                .detach()
                .float()
                .item()
            )


            if aux_loss is not None:

                aux_loss_sum += (
                    aux_loss
                    .detach()
                    .float()
                    .item()
                )


        result[
            split
        ] = (
            total_loss_sum
            / eval_iters
        )


        result[
            f"{split}_lm"
        ] = (
            lm_loss_sum
            / eval_iters
        )


        result[
            f"{split}_aux"
        ] = (
            aux_loss_sum
            / eval_iters
        )


    if was_training:

        model.train()


    return result


# ============================================================
# CHECKPOINT SAVE
# ============================================================

def save_checkpoint(
    path,
    model,
    optimizer,
    scaler,
    step,
    best_val_loss,
):

    checkpoint = {
        "model":
            model.state_dict(),

        "optimizer":
            optimizer.state_dict(),

        "step":
            step,

        "config":
            cfg,

        "best_val_loss":
            best_val_loss,

        "data_rng_state":
            DATA_RNG.bit_generator.state,

        "torch_rng_state":
            torch.get_rng_state(),

        "python_rng_state":
            random.getstate(),

        "scaler":
            (
                scaler.state_dict()
                if scaler is not None
                else None
            ),
    }


    if device == "cuda":

        checkpoint[
            "cuda_rng_state_all"
        ] = torch.cuda.get_rng_state_all()


    temporary_path = (
        path
        + ".tmp"
    )


    torch.save(
        checkpoint,
        temporary_path,
    )


    os.replace(
        temporary_path,
        path,
    )


# ============================================================
# TRAINING
# ============================================================

print()
print("=" * 70)
print("STARTING V4 TRAINING")
print("=" * 70)


t0 = time.time()

running_mfu = None


for step in range(
    start_step,
    PRETRAIN_MAX_STEPS,
):


    # ========================================================
    # RESET PEAK VRAM STATISTICS
    # ========================================================

    if device == "cuda":

        torch.cuda.reset_peak_memory_stats()


    # ========================================================
    # LEARNING RATE
    # ========================================================

    lr = get_lr(
        step,
        PRETRAIN_WARMUP_STEPS,
        PRETRAIN_MAX_STEPS,
        PRETRAIN_MAX_LR,
        PRETRAIN_MIN_LR,
    )


    for param_group in (
        optimizer.param_groups
    ):

        param_group[
            "lr"
        ] = lr


    # ========================================================
    # CLEAR GRADIENTS
    # ========================================================

    optimizer.zero_grad(
        set_to_none=True
    )


    accumulated_total_loss = 0.0
    accumulated_lm_loss = 0.0
    accumulated_aux_loss = 0.0


    # ========================================================
    # MICRO-STEPS
    # ========================================================

    for micro_step in range(
        PRETRAIN_GRAD_ACCUM
    ):

        x, y = get_batch(
            "train",
            PRETRAIN_BLOCK_SIZE,
            PRETRAIN_BATCH_SIZE,
            device,
        )


        # ----------------------------------------------------
        # AUTOCast
        # ----------------------------------------------------

        if device == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=ptdtype,
            ):

                (
                    _,
                    total_loss,
                    lm_loss,
                    aux_loss,
                ) = forward_train(
                    compiled_model,
                    x,
                    y,
                )

        else:

            (
                _,
                total_loss,
                lm_loss,
                aux_loss,
            ) = forward_train(
                compiled_model,
                x,
                y,
            )


        # ----------------------------------------------------
        # FINITE LOSS CHECK
        # ----------------------------------------------------

        if not torch.isfinite(
            total_loss
        ):

            raise RuntimeError(
                f"Non-finite loss at "
                f"step={step}, "
                f"micro_step={micro_step}: "
                f"{total_loss.item()}"
            )


        # ----------------------------------------------------
        # GRADIENT ACCUMULATION
        # ----------------------------------------------------

        scaled_loss = (
            total_loss
            / PRETRAIN_GRAD_ACCUM
        )


        # ----------------------------------------------------
        # BACKWARD
        # ----------------------------------------------------

        if scaler is not None:

            scaler.scale(
                scaled_loss
            ).backward()

        else:

            scaled_loss.backward()


        # ----------------------------------------------------
        # LOGGING ACCUMULATED LOSSES
        # ----------------------------------------------------

        accumulated_total_loss += (
            total_loss
            .detach()
            .float()
            .item()
            / PRETRAIN_GRAD_ACCUM
        )


        accumulated_lm_loss += (
            lm_loss
            .detach()
            .float()
            .item()
            / PRETRAIN_GRAD_ACCUM
        )


        if aux_loss is not None:

            accumulated_aux_loss += (
                aux_loss
                .detach()
                .float()
                .item()
                / PRETRAIN_GRAD_ACCUM
            )


        del x
        del y
        del total_loss
        del lm_loss
        del scaled_loss

        if aux_loss is not None:
            del aux_loss


    # ========================================================
    # UNSCALE FP16 GRADIENTS
    # ========================================================

    if scaler is not None:

        scaler.unscale_(
            optimizer
        )


    # ========================================================
    # GRADIENT CLIPPING + FINITENESS
    # ========================================================

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        PRETRAIN_GRAD_CLIP,
        error_if_nonfinite=False,
    )


    if not torch.isfinite(
        grad_norm
    ):

        print()
        print("=" * 70)
        print("NON-FINITE GRADIENT")
        print("=" * 70)

        print(
            f"step         : {step}"
        )

        print(
            f"learning rate: {lr:.3e}"
        )

        print(
            "optimizer step SKIPPED"
        )

        print("=" * 70)


        optimizer.zero_grad(
            set_to_none=True
        )


        if scaler is not None:
            scaler.update()


        continue


    # ========================================================
    # TOP GRADIENT NORMS
    # ========================================================

    top_grad_norms = []


    if step % 10 == 0:

        for (
            name,
            parameter,
        ) in model.named_parameters():

            if parameter.grad is None:
                continue


            norm = (
                parameter.grad
                .detach()
                .float()
                .norm()
                .item()
            )


            top_grad_norms.append(
                (
                    name,
                    norm,
                )
            )


        top_grad_norms.sort(
            key=lambda x: -x[1]
        )


    # ========================================================
    # OPTIMIZER STEP
    # ========================================================

    if scaler is not None:

        scaler.step(
            optimizer
        )

        scaler.update()

    else:

        optimizer.step()


    # ========================================================
    # CUDA SYNCHRONIZATION FOR TIMING
    # ========================================================

    if device == "cuda":

        torch.cuda.synchronize()


    dt = (
        time.time()
        - t0
    )

    t0 = time.time()


    # ========================================================
    # TOKENS / SEC
    # ========================================================

    tokens_this_step = (
        PRETRAIN_BATCH_SIZE
        * PRETRAIN_GRAD_ACCUM
        * PRETRAIN_BLOCK_SIZE
    )


    tokens_per_sec = (
        tokens_this_step
        / max(dt, 1e-9)
    )


    # ========================================================
    # MFU
    # ========================================================

    mfu = None


    if hasattr(
        model,
        "estimate_mfu",
    ):

        try:

            mfu = model.estimate_mfu(
                PRETRAIN_BATCH_SIZE
                * PRETRAIN_GRAD_ACCUM,
                dt,
                peak_flops=PEAK_FLOPS,
            )

        except Exception:

            mfu = None


    if mfu is not None:

        if running_mfu is None:

            running_mfu = mfu

        else:

            running_mfu = (
                0.9
                * running_mfu
                + 0.1
                * mfu
            )


    # ========================================================
    # PEAK VRAM
    # ========================================================

    peak_vram_mb = None


    if device == "cuda":

        peak_vram_mb = (
            torch.cuda.max_memory_allocated()
            / (
                1024 ** 2
            )
        )


    # ========================================================
    # PRINT TRAINING STATUS
    # ========================================================

    message = (
        f"step {step:6d} | "
        f"total {accumulated_total_loss:.4f} | "
        f"lm {accumulated_lm_loss:.4f} | "
        f"aux {accumulated_aux_loss:.4f} | "
        f"lr {lr:.2e} | "
        f"grad {float(grad_norm):.3f} | "
        f"{tokens_per_sec:,.0f} tok/s"
    )


    if running_mfu is not None:

        message += (
            f" | mfu "
            f"{running_mfu * 100:.1f}%"
        )


    if peak_vram_mb is not None:

        message += (
            f" | VRAM "
            f"{peak_vram_mb:.0f} MB"
        )


    print(message)


    # ========================================================
    # TOP GRADIENT DIAGNOSTICS
    # ========================================================

    if (
        step % 10 == 0
        and top_grad_norms
    ):

        print(
            "  top grad norms:",
            [
                (
                    name,
                    round(
                        value,
                        3,
                    ),
                )
                for (
                    name,
                    value,
                ) in top_grad_norms[:5]
            ],
        )


    # ========================================================
    # VALIDATION
    # ========================================================

    if (
        step > 0
        and
        step
        % PRETRAIN_EVAL_INTERVAL
        == 0
    ):

        print()
        print(
            "Running validation..."
        )


        eval_results = estimate_loss(
            model,
            PRETRAIN_BLOCK_SIZE,
            PRETRAIN_BATCH_SIZE,
            device,
            ptdtype,
            PRETRAIN_EVAL_ITERS,
        )


        print(
            f"  train total : "
            f"{eval_results['train']:.4f}"
        )

        print(
            f"  train LM    : "
            f"{eval_results['train_lm']:.4f}"
        )

        print(
            f"  train aux   : "
            f"{eval_results['train_aux']:.4f}"
        )

        print(
            f"  val total   : "
            f"{eval_results['val']:.4f}"
        )

        print(
            f"  val LM      : "
            f"{eval_results['val_lm']:.4f}"
        )

        print(
            f"  val aux     : "
            f"{eval_results['val_aux']:.4f}"
        )


        # ----------------------------------------------------
        # BEST CHECKPOINT
        # ----------------------------------------------------

        if (
            eval_results["val"]
            < best_val_loss
        ):

            best_val_loss = (
                eval_results["val"]
            )


            save_checkpoint(
                pretrain_best_ckpt_path,
                model,
                optimizer,
                scaler,
                step,
                best_val_loss,
            )


            print(
                f"  ✅ new best "
                f"val total: "
                f"{best_val_loss:.4f}"
            )


    # ========================================================
    # LATEST CHECKPOINT
    # ========================================================

    if (
        step > 0
        and
        step
        % PRETRAIN_CKPT_INTERVAL
        == 0
    ):

        save_checkpoint(
            pretrain_ckpt_path,
            model,
            optimizer,
            scaler,
            step,
            best_val_loss,
        )


        print(
            f"  checkpoint saved:"
            f" {pretrain_ckpt_path}"
        )


# ============================================================
# FINAL CHECKPOINT
# ============================================================

final_step = max(
    start_step,
    PRETRAIN_MAX_STEPS - 1,
)


save_checkpoint(
    pretrain_ckpt_path,
    model,
    optimizer,
    scaler,
    final_step,
    best_val_loss,
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 70)
print("V4 PRETRAINING FINISHED")
print("=" * 70)

print(
    f"best validation total loss : "
    f"{best_val_loss:.4f}"
)

print(
    f"best checkpoint            : "
    f"{pretrain_best_ckpt_path}"
)

print(
    f"latest checkpoint          : "
    f"{pretrain_ckpt_path}"
)

print("=" * 70)


V4 PRETRAINING SETUP
device                  : cuda
GPU                     : NVIDIA GeForce RTX 5050 Laptop GPU
dtype                   : torch.bfloat16
GradScaler              : False
micro batch             : 1
grad accumulation       : 32
effective batch         : 32
sequence length         : 1024
tokens/update           : 32,768
max steps               : 152,588
max training tokens     : 5,000,003,584
torch.compile           : False
checkpoint debug        : False

LOADING TRAINING SHARDS
training shards       : 500
training tokens       : 5,000,000,000
training tokens       : 5.000B
validation tokens    : 245,104

Testing data loader...
train shape  : (1, 1024)
train dtype  : torch.int64
val shape    : (1, 1024)
✅ Data loader sanity check PASSED

CREATING V4 MODEL
total parameters      : 165.91M
non-embedding params  : 122.49M
AdamW fused           : True

RESUMING FROM CHECKPOINT
C:\Users\sandyarjun\gpt150m_project\data\pretrain_checkpoints\latest.pt


TypeError: RNG state must be a torch.ByteTensor

## 6. Sanity check: base model text continuation
This is the *base* model (not instruction-tuned yet) -- it continues text, it does not answer questions yet.

In [ ]:
# Standalone sanity check: loads the PRETRAINED (base) checkpoint fresh from
# disk and generates from it. Run this after Sections 1-2 (setup + model def)
# have been run in the notebook, so GPTConfig/GPT/enc/EOT/device/cfg exist.
#
# This is a base (not instruction-tuned) model -- it will just continue text,
# it does not yet know how to "answer" a question. That's expected and fine;
# we're only checking that pretraining produced something coherent.

pretrain_best_ckpt_path = os.path.join(PRETRAIN_CKPT_DIR, 'best.pt')
pretrain_ckpt_path = os.path.join(PRETRAIN_CKPT_DIR, 'latest.pt')
ckpt_path = pretrain_best_ckpt_path if os.path.exists(pretrain_best_ckpt_path) else pretrain_ckpt_path

test_model = GPT(cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
test_model.load_state_dict(ckpt['model'])
test_model.eval()
print(f"testing pretrain checkpoint from step {ckpt['step']} ({ckpt_path})")
print(f"best_val_loss recorded: {ckpt.get('best_val_loss', 'n/a')}")
print()

prompts = [
    'Once upon a time',
    'The scientist looked at the data and',
    'In the middle of the forest,',
    'The capital of France is',
]

for prompt in prompts:
    prompt_ids = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long).to(device)
    out = test_model.generate(
        prompt_ids, max_new_tokens=80, temperature=0.8,
        top_k=50, top_p=0.9, repetition_penalty=1.3, eos_token_id=EOT,
    )
    print(f'PROMPT: {prompt}')
    print(enc.decode(out[0].tolist()))
    print()

testing pretrain checkpoint from step 26600 (C:\Users\sandyarjun\gpt150m_project\checkpoints_pretrain\best.pt)
best_val_loss recorded: 1.4497102499008179

PROMPT: Once upon a time
Once upon a time, there was an old lady. She had very long hair and loved to take care of her comb. One day she went for a walk in the park with her comb. As she walked, she saw lots of children playing. 

The old lady stopped and said, "Hello! What are you doing?" The kids replied, "We're playing hide-and-seek!" The old

PROMPT: The scientist looked at the data and
The scientist looked at the data and smiled. He was not angry anymore, but he had a problem with his work. His computer made strange noises and went all over the room very fast!

The family laughed and said they would try to fix it soon so that Tim could learn how to use computers properly again in one day.<|endoftext|>

PROMPT: In the middle of the forest,
In the middle of the forest, there was a big tree. In the trunk, there were many fruit. A l

In [ ]:
# Quick sanity check: this is a base (not instruction-tuned) model, so it will
# just continue text -- it does not yet know how to "answer" a question.
# Uses top_k + top_p (nucleus) sampling together with a repetition penalty,
# which meaningfully reduces the looping/repeating text that small models
# are prone to.
model.eval()
prompt = 'Once upon a time'
prompt_ids = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long).to(device)
out = model.generate(
    prompt_ids, max_new_tokens=80, temperature=0.8,
    top_k=50, top_p=0.9, repetition_penalty=1.3, eos_token_id=EOT,
)
print(enc.decode(out[0].tolist()))
model.train()

Once upon a time, there was a little girl named Lily. She had an adorable puppy who she loved very much. One day, Lily and her mommy went to the park to play with their friends. While they were playing, Lily's friend Billy came over and asked if he could lend his toy car. 

Lily thought about it for a moment before saying no because she didn't want someone else


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 864)
    (wpe): Embedding(512, 864)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((864,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=864, out_features=2592, bias=True)
          (c_proj): Linear(in_features=864, out_features=864, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((864,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=864, out_features=3456, bias=True)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=3456, out_features=864, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((864,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head):